# ComfyUI Fixed Colab

Run cells from top to bottom. This fixed notebook installs or updates ComfyUI on Google Drive, installs ComfyUI-Manager, fixes dependency issues like `alembic`, removes stale files that can cause `comfy_aimdo` import errors, downloads SD1.5 + VAE, and launches ComfyUI through Cloudflare Tunnel.

## 1. Check GPU

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=False)


## 2. Install or update ComfyUI

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess

USE_GOOGLE_DRIVE = True
CLEAN_STALE_FILES = True
WORKSPACE = Path('/content/drive/MyDrive/ComfyUI') if USE_GOOGLE_DRIVE else Path('/content/ComfyUI')

def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)

if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')

WORKSPACE.parent.mkdir(parents=True, exist_ok=True)

if not WORKSPACE.exists():
    run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', str(WORKSPACE)])

if CLEAN_STALE_FILES:
    stale_paths = [
        WORKSPACE / 'comfy_aimdo',
        WORKSPACE / 'comfy_aimdo.py',
        WORKSPACE / 'comfy_api_nodes',
    ]
    for path in stale_paths:
        if path.is_dir():
            shutil.rmtree(path)
            print('Removed stale directory:', path)
        elif path.exists():
            path.unlink()
            print('Removed stale file:', path)

run(['git', 'reset', '--hard', 'HEAD'], cwd=WORKSPACE)
run(['git', 'pull', '--ff-only'], cwd=WORKSPACE)

run(['python3', '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run(['python3', '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121'])
run(['python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=WORKSPACE)
run([
    'python3', '-m', 'pip', 'install', '-q',
    'alembic', 'blake3', 'GitPython', 'accelerate', 'einops', 'transformers',
    'safetensors', 'aiohttp', 'pyyaml', 'Pillow', 'scipy', 'tqdm', 'psutil',
    'tokenizers', 'torchsde', 'kornia', 'spandrel', 'soundfile', 'sentencepiece'
])

print('ComfyUI setup complete:', WORKSPACE)


## 3. Install or update ComfyUI-Manager

In [ ]:
from pathlib import Path
import subprocess

WORKSPACE = Path('/content/drive/MyDrive/ComfyUI')
custom_nodes = WORKSPACE / 'custom_nodes'
manager = custom_nodes / 'ComfyUI-Manager'
custom_nodes.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)

if not manager.exists():
    run(['git', 'clone', 'https://github.com/ltdrdata/ComfyUI-Manager.git', str(manager)])

run(['git', 'reset', '--hard', 'HEAD'], cwd=manager)
run(['git', 'pull', '--ff-only'], cwd=manager)
run(['python3', 'custom_nodes/ComfyUI-Manager/cm-cli.py', 'restore-dependencies'], cwd=WORKSPACE)

print('ComfyUI-Manager setup complete')


## 4. Download starter models

In [ ]:
from pathlib import Path
import subprocess

WORKSPACE = Path('/content/drive/MyDrive/ComfyUI')
(WORKSPACE / 'models' / 'checkpoints').mkdir(parents=True, exist_ok=True)
(WORKSPACE / 'models' / 'vae').mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)

run([
    'wget', '-c',
    'https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt',
    '-P', str(WORKSPACE / 'models' / 'checkpoints')
])
run([
    'wget', '-c',
    'https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors',
    '-P', str(WORKSPACE / 'models' / 'vae')
])


## 5. Launch with Cloudflare Tunnel

Wait until the output shows `ComfyUI URL: https://...trycloudflare.com`. Open that link.

In [ ]:
from pathlib import Path
import re
import socket
import subprocess
import threading
import time

WORKSPACE = Path('/content/drive/MyDrive/ComfyUI')

def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)

run(['wget', '-q', '-O', '/content/cloudflared-linux-amd64.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'])
run(['dpkg', '-i', '/content/cloudflared-linux-amd64.deb'], check=False)

def launch_cloudflared(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break
    print('ComfyUI loaded. Starting Cloudflare Tunnel...')
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    for line in proc.stderr:
        match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
        if match:
            print('ComfyUI URL:', match.group(0))

threading.Thread(target=launch_cloudflared, daemon=True, args=(8188,)).start()
run(['python3', 'main.py', '--dont-print-server'], cwd=WORKSPACE)
